# ST554 Final Project: Siona Benjamin
For this final project we will use Spark to handle streaming data and fitting a machine learning model. The data set in question describes power consumption from different zones of Tetoauan City in relation to factors such as time of day, temperature, and humidity. Our goal is to create a model that can predict the power consumption from a particular zone based off other variables in the dataset. This is beneficial if the measurement for a certain zone goes offline and needs to be determined another way. Once we create our model, we also want to make predictions about the power consumption in this zone in real time as new data is recieved. For this, we can take advantage of Spark's streaming capabilities. 

To get started, we of course need to import the necessary modules. Then, we'll read in our data as a pandas dataframe before converting this to a spark dataframe.

In [14]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, CrossValidatorModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import LinearRegression
from pyspark.sql.types import StructType
from pyspark.sql.functions import col
from pyspark.ml.feature import SQLTransformer, PCA, Binarizer, OneHotEncoder, VectorAssembler, StringIndexer

In [15]:
#create spark session
spark = SparkSession.builder.appName("final_project").getOrCreate()

In [36]:
#import data as pandas dataframe
power_data = pd.read_csv('power_ml_data.csv')
#convert pandas dataframe to spark dataframe
power_df = spark.createDataFrame(power_data)

Using `.show()` we can see what our data columns look like while `.dtypes` lets us see what data type each column is stored as. We see that most values are stored as doubles except for the Month and Hour variables.

In [17]:
power_df.show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

In [18]:
power_df.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

## Fitting the Model
The first part of this project will be training an elastic net model with out dataset to predict power consumption values for Zone 3. In an elastic net model, L1 (LASSO) and L2 (Ridge) penalties are combined to improve model predictions and stability. 

Now that we've loaded our dataset and have a good idea of what our data looks like, we can set up the transformations we want to apply to our data before training our model. The first transformation we'll apply is a SQL transformation to cast the Hour variable as a double instead of an integer. Within the same transformation, we will also rename the Power_Zone_3 column as label. After changing the Hour variable type, we'll apply a binarizer transformation to this variable to distinguish between night and day using 6.5 as the cutoff. Next , we'll use one-hot encoding to encode the Month variable. Additionally, we will run a PCA (Principle Component Analysis) fit on a few of the columns in our dataset. The PCA entails using a VectorAssembler transformation to place the desired variables together in a column followed by using the PCA transformation.

Lastly, we will use the VectorAssembler transformation to combine our desired predictor variables in a features column. 

In [6]:
#SQL transformer to cast Hour variable as DoubleType
sqlTrans = SQLTransformer(
    statement = """
                SELECT *,
                CAST(Hour AS DOUBLE) AS hour_double,
                Power_Zone_3 as label 
                FROM __THIS__
                """)

In [7]:
#Binarize transformer to convert continuous Hour values to binary values 
binarizer = Binarizer(threshold=6.5, inputCol="hour_double", outputCol="hour_binary")

In [8]:
#One-hot encoder to transform Month values to vector values
##StringIndexer transformation to conver Month values 
indexer = StringIndexer(inputCol="Month", outputCol="month_index")
##OneHotEncoder transformation
encoder = OneHotEncoder(inputCols=["Month"], outputCols=["month_vec"])

In [9]:
#PCA transformation 
##VectorAssembler to combine desired columns
pca_assembler = VectorAssembler(inputCols=["Temperature","Humidity","Wind_Speed","General_Diffuse_Flows","Diffuse_Flows"], outputCol="pca_features")
##PCA transformer 
pca = PCA(k=2,inputCol="pca_features", outputCol="pca_results")

In [10]:
#VectorAssembler to put predictors in features column 
assembler_features = VectorAssembler(inputCols=["hour_binary","Power_Zone_1","Power_Zone_2","month_vec","pca_results"], outputCol="features")

Now that we have defined our transformations, we can define the other components of our model. First we'll create an object to define our linear regression model. Then we'll define our parameter grid to set test values of `regParam`, which controls the amount of regularization, and `elasticNetParam`, which defines the balance between L1 and L2 regularization. We will also set up a pipeline with the transformations defined above and our linear regression model. 

In [11]:
#define object for linear regression model 
lr = LinearRegression()
#define parameter grid 
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()
#define transformation pipeline 
pipeline = Pipeline(stages = [sqlTrans, binarizer, indexer, encoder, pca_assembler, pca, assembler_features, lr])

The next step is to set up our `CrossValidator` object and enter in our pipeline, parameter grid, and RMSE regression evaluator. For our cross validation, we'll use 5 folds. Now we can fit our cross validation model.

In [12]:
#set up cross validation 
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [13]:
cvModel = crossval.fit(power_df)

26/04/28 17:10:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/28 17:10:07 WARN Instrumentation: [b6cdef7a] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:09 WARN Instrumentation: [b6cdef7a] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 17:10:11 WARN Instrumentation: [148b699c] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:12 WARN Instrumentation: [148b699c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 17:10:14 WARN Instrumentation: [198ef797] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:14 WARN Instrumentation: [198ef797] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

At this point, we've successfully used cross validation to train an elastic net model! Let's save this model to avoid having to rerun the training every time we reopen our notebook. We can use `.save()` to save our model in a folder, and `.load()` to reload this model when we want to.

In [14]:
#save model in directory
cvModel.write().overwrite().save("cvModel")

In [19]:
#load model from directory
cvModel = CrossValidatorModel.load("cvModel")

Let's see how our elastic net model performs. After cross validation, the optimal hyperparameters chosen were a `regParam` of 0.05 and an `elasticNetParam` of 0.1. 

In [61]:
#extract last training stage of the best model 
best_model = cvModel.bestModel.stages[-1]
#iterate through paramters in parameter map for the best model
print("Optimal Paramters:")
print("-" * 30)
for param, value in best_model.extractParamMap().items():
    #print regParam and elasticNetParam
    if param.name in [p.name for p in paramGrid[0].keys()]:
        print(f"{param.name}: {value}")

Optimal Paramters:
------------------------------
elasticNetParam: 0.1
regParam: 0.05


We can also take a look at the CV errors for each combination of `regParam` and `elasticNetParam`. We can see that many of the RMSE values eneded up being similar varying only in their decimal point values.

In [39]:
print("CV Errors:")
print("-" * 30)
for params, score in zip(paramGrid, cvModel.avgMetrics):
    param_str = "|".join([f"{param.name}={value}" for param, value in params.items()])
    print(f"{param_str} -- RMSE: {score:.4f}")

CV Errors:
------------------------------
regParam=0.0|elasticNetParam=0.0 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.05 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.1 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.25 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.5 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.75 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.9 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.95 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.98 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.99 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=1.0 -- RMSE: 2147.8759
regParam=0.05|elasticNetParam=0.0 -- RMSE: 2147.8758
regParam=0.05|elasticNetParam=0.05 -- RMSE: 2147.8768
regParam=0.05|elasticNetParam=0.1 -- RMSE: 2147.8751
regParam=0.05|elasticNetParam=0.25 -- RMSE: 2147.8762
regParam=0.05|elasticNetParam=0.5 -- RMSE: 2147.8753
regParam=0.05|elasticNetParam=0.75 -- RMSE: 2147.8756
regParam=0.05|elasticNetParam=0.9 -- RMSE: 2147.8755
regPar

Now we also want to calculate the resulting RMSE when we use our cvModel to predict Power_Zone_3 values of our original dataset. Doing so gives us an RMSE value of 2147.097. 

In [40]:
lr_rmse = RegressionEvaluator().evaluate(cvModel.transform(power_df))
print(f"RMSE: {lr_rmse}")

RMSE: 2147.0973169293934


Our last step with our model will be using it to add a residual column to our dataframe. First, we use our cvModel as a transformation to add a column of predicted values to our dataframe. We also create a column with residuals values showing the deviation of our predicted values from the original values. 

In [15]:
power_df_pred = cvModel.transform(power_df)
power_df_resid = power_df_pred.withColumn("residual",col("label")-col("prediction"))
power_df_resid.select("label","prediction","residual").show(8)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20878.850660788765| -637.886800788765|
|20131.08434|18660.227266544003| 1470.857073455998|
|19668.43373| 18204.75215311452|1463.6815768854794|
|18899.27711|17590.648498339124|1308.6286116608753|
|18442.40964| 16997.30198645687|1445.1076535431312|
|18130.12048| 16517.68672349429|1612.4337565057103|
|17945.06024|16093.246141053492|1851.8140989465064|
|17459.27711|15722.695360253929|1736.5817497460703|
+-----------+------------------+------------------+
only showing top 8 rows


Now that we have successfully trained our elastic net model, we can move onto making predictions with new data.

## Streaming Data
In the previous section we trained an elastic net model on our dataset to predict the power consumption of Zone 3. With this model, we can now make predictions with new data that we read in from a stream. First we define a schema for the data that will be streamed in, following the schema from our `power_df` dataframe we used in the previous section. Next, we set up our stream using `.readStream()` and tell the stream to look for new data in the folder streaming_files. 

In [37]:
#define schema for read in data
myschema = power_df.schema

In [38]:
#read csv files from folder 'streaming_files' following schema 
stream_df = spark.readStream.schema(myschema).format("csv").option("header","true").load("streaming_files")

We also want to set up some tranformations to process our read in data. First, we'll rename the Power_Zone_3 column to label. Then we'll use our previously trained cvModel to predict values for Power_Zone_3 in our new data. At the same time, we'll also calculate the residuals for predicted values and select only the prediction, residual, and label columns from this dataframe. Once we've defined the transformations we want to do on our stream, we can use `.join()` to combine these dataframes and print them together to the console. Then, we'll use `.writeStream()` to write our transformed data stream to the console.

In [39]:
#transformation to rename response variable to label
rename_df = stream_df.withColumnRenamed("Power_Zone_3","label") 

#transformation to predict Power_Zone_3, create residual, and select columns
predict_df = cvModel.transform(stream_df).withColumn("residual",col("label")-col("prediction")).select("prediction", "residual","label")

#join rename_df and predict_df
joined_df=rename_df.join(predict_df, on='label')

#write stream to console
writeDF = joined_df.writeStream.outputMode("append").format("console").start()

26/04/29 23:13:43 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c5bfa080-bddc-486f-821a-8d22657b0442. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/29 23:13:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


At this point we have started our data stream and have definined which transormations we want to make to our data before it is written to the console. In a python file named `produce_stream_data.py`, we have a script that samples 5 records from a larger database and saves them in a csv file for us to read using our stream. Now that we have our stream read and write set up, we can run `produce_stream_data.py` to populate our streaming_files folder with csv files that will be read by the stream.

Once we've read in all of our data, we can stop writing to the console.

In [40]:
%run produce_stream_data.py

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|          residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+------------------+
|14869.78723|      18.61|    84.2|     0.085|                0.069|        0.089| 33973.91685| 20404.97925|   10|  23| 13730.76105818801|1139.0261718119891|
| 18702.6506|      19.75|   67.34|     4.918|                445.5|         87.4| 30215.38462| 22243.38843|   11|  14|12211.868646946408| 6490.781953053593|
|16321.93548|      18.95|   69.16|     0.082|                680.5|        54.75|  32801.3617|  23140.2439|    3|  12|

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
| 12990.6383|      22.64|   68.37|      4.92|                136.5|         94.9| 35316.23632| 22604.56432|   10|  14|14210.109197916117|-1219.4708979161169|
|10014.88595|      20.18|   27.47|     0.085|                 66.6|        68.16| 32285.93156|  26691.6232|   12|  16|12819.068115606937| -2804.182165606937|
|33240.50209|      23.86|    83.9|     4.903|                0.069|        0.141| 41174.75083| 25743.03797|    7|

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|27578.18182|      25.96|   67.89|     0.067|                820.0|         70.1| 40422.28635| 28199.36642|    8|  13| 25659.07564148528| 1919.1061785147213|
|25321.12853|       25.6|   55.89|     4.925|                662.4|        161.8| 41246.97003| 27146.35692|    8|  12|26125.984496326477|  -804.855966326475|
|11368.42105|      15.96|    78.7|     0.076|                50.87|        45.09| 24336.78689| 15339.93808|    5|

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|16768.40125|      25.05|    93.0|     4.909|                0.161|        0.115| 24190.72142| 16167.68743|    8|   6|18284.347872490802|-1515.9466224908028|
|15249.45455|      13.89|    80.7|      0.08|                0.029|        0.185| 23474.01507| 13993.07536|    4|   4|15369.049392463068|-119.59484246306783|
|21285.94975|      14.16|   66.49|     0.086|                0.044|        0.111| 35304.40678| 21465.04559|    2|

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|15030.36145|      14.36|   68.86|     0.077|                0.081|        0.156| 24322.02532|   15643.769|    1|   6|14931.530055283274|  98.83139471672621|
|24150.36145|      10.76|    80.5|     0.086|                0.037|        0.156| 39451.13924| 24959.27052|    1|  22| 23339.18084534652|  811.1806046534803|
|10877.42714|      6.523|   56.53|     0.091|                0.033|         0.13| 18170.84746|  10734.3465|    2|

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|          residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+------------------+
|25257.16583|      12.87|   65.65|     0.082|                0.073|        0.096| 41772.20339| 26071.73252|    2|  21| 24659.02420373865| 598.1416262613529|
|9317.647059|       9.13|    85.0|     0.083|                0.048|        0.156| 20720.91255| 16425.89751|   12|   4| 6811.767945829043|2505.8791131709577|
|15062.83417|      13.95|   56.17|     4.919|                0.084|        0.119| 23717.28814| 14097.26444|    2|   2

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|          residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+------------------+
|    19136.0|      21.07|   44.45|     0.089|                854.0|        52.96|  36060.4521| 21358.04481|    4|  12|20007.874750156567|-871.8747501565667|
|23302.34818|      22.68|   60.44|      0.08|                0.073|        0.082|  39621.2459| 24334.36533|    5|  23| 22892.94074541718| 409.4074345828194|
|24764.51613|      13.69|    73.1|     0.083|                0.055|        0.126| 42464.68085| 25693.90244|    3|  21

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|11671.73252|      22.01|    76.5|     4.924|                572.9|         93.6| 36022.05689| 24296.26556|   10|  11|13870.905728278609|-2199.1732082786093|
|14684.51613|      12.04|    79.0|     0.087|                112.1|        105.4| 29535.31915| 16982.92683|    3|   9|15402.290451351175| -717.7743213511749|
|25054.83871|      17.14|    76.3|     4.914|                0.911|        0.931| 44480.68085| 26436.58537|    3

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|28469.16923|      23.58|    45.7|     4.917|                0.102|        0.115| 46194.43709| 27490.22869|    6|  21|28570.907661260244| -101.7384312602444|
|20142.54545|      16.74|    68.1|     0.081|                0.044|        0.108| 30765.46825| 17494.09369|    4|  23|18402.859371990813| 1739.6860780091883|
|17402.18182|      18.24|    78.2|     0.077|                319.6|        269.4| 35806.24327| 17358.45214|    4

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|          residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+------------------+
|24892.25806|      13.67|   51.13|     0.086|                0.242|        0.204| 42758.80851| 25785.36585|    3|  19| 24901.96945602196|-9.711396021961264|
|16345.16129|      11.45|    88.5|      0.08|                65.36|        60.46| 31483.91489| 18863.41463|    3|  16|17016.138020477705|-670.9767304777051|
| 11070.6383|      13.09|    86.9|     4.915|                0.048|        0.104| 24161.75055| 19236.09959|   10|   5

In [41]:
writeDF.stop()

26/04/29 23:22:04 WARN DAGScheduler: Failed to cancel job group b4f5e739-d234-4568-8916-d7f7a67f64e5. Cannot find active jobs for it.
26/04/29 23:22:04 WARN DAGScheduler: Failed to cancel job group b4f5e739-d234-4568-8916-d7f7a67f64e5. Cannot find active jobs for it.


## Conclusion
In this project, we trained an elastic net model with pyspark MLlib to predict the power consumption for a certain zone in Tetoauan City from the other columns in the dataset such as temperature, wind speed, and measurements for other zones. Using cross validation resulted in an elastic net model with a regularization of 0.05 and an elastic net paramter of 0.1. Once we trained our model, we were then able to apply the model as a tranformation to make predictions with new data. We did this by setting up a stream to read in new data from our streaming_files folder. We also applied other transformation to the streamed in data, such as calculating the residual and relabeling column names to allow the joining of two separate dataframes. 

With this work, we see how to train a machine learned model with pyspark MLlib. We also saw how to set up a stream to efficiently read in and make predictions with new data.